In [21]:
from collections import defaultdict
import gymnasium as gym
import numpy as np

env = gym.make("CartPole-v0")
state_size = env.observation_space.shape[0]  # Size of the state space
action_size = env.action_space.n  # Number of possible actions

## Actor/Critic Networks

In RL, the Actor-Critic architecture is commonly used to train agents. The Actor network is responsible for selecting actions based on the current game state, and the Critic is responsible for scoring those actions. The two networks are trained together and improve each other over time.

This Actor network is a simple three-layer feedforward neural network that takes in the current game state and outputs the probabilities of each possible action.

In [31]:
from torch.distributions import Categorical
import torch.nn as nn
import torch.nn.functional as F

class Actor(nn.Module):
    def __init__(self, state_size, action_size):
        super(Actor, self).__init__()
        self.state_size = state_size
        self.action_size = action_size
        self.linear1 = nn.Linear(self.state_size, 128)
        self.linear2 = nn.Linear(128, 256)
        self.linear3 = nn.Linear(256, self.action_size)

    def forward(self, state):
        l1_res = F.relu(self.linear1(state))
        l2_res = F.relu(self.linear2(l1_res))
        l3_res = F.relu(self.linear3(l2_res))
        # we use categorical distribution to help wrap our probability distributions in some helpful tools that we'll use more later
        distribution = Categorical(F.softmax(l3_res, dim=-1))
        return distribution

In [36]:
class Critic(nn.Module):
    def __init__(self, state_size):
        super(Critic, self).__init__()
        self.state_size = state_size
        self.linear1 = nn.Linear(self.state_size, 128)
        self.linear2 = nn.Linear(128, 256)
        self.linear3 = nn.Linear(256, 1)

    def forward(self, state):
        l1_res = F.relu(self.linear1(state))
        l2_res = F.relu(self.linear2(l1_res))
        l3_res = self.linear3(l2_res)
        return l3_res

The Critic network is nearly identical, but it outputs a single value--the expected value of the state.

## PPO Agent

I used Generalized Advantage Estimation (GAE) to compute the advantage function. GAE is the method used in the PPO paper. GAE takes in the realized rewards for each step, the predicted rewards for each step, and some hyperparameters. Then, it calculates the difference between the realized and predicted rewards (with some decay). The delta is then used to compute the GAE, which is the delta plus the discounted GAE from the next step. Think of the GAE as the carry-through reward for the next step; if the next step is better than expected, then the current step is also better than expected.

Knowing what the decay on the delta is isn't hugely important to PPO, but a good mental model is that the further in the future some reward is, the less likely the current step is to be responsible for it. Your being able to cut a straight line in preschool probably has less of a bearing on whether you will get into MIT. From a math standpoint, we add the next step's GAE * gamma * lam. Gamma is the discount rate for future rewards (think about cutting and MIT), and lambda (lam) is the decay for the GAE. Lambda determines much a model should trust it's own predictions; 0 means it trusts itself a lot, and 1 means it doesn't trust itself at all. The PPO paper uses 0.95, which is a good balance between trusting the model and trusting the environment.

In [19]:
def compute_gae(next_value, rewards, masks, values, gamma=0.99, lam=0.95):
    advantages = []
    gae = 0
    for step in reversed(range(len(rewards))):
        nv = next_value if step == len(rewards) - 1 else values[step + 1]
        # we use the decay (reward[step] + gamma * nv * mask[step]) to weight a future action less.
        delta = rewards[step] + gamma * nv * masks[step] - values[step]
        # gae adds a "carry-through" reward to advantages, which is weighted
        # basically: how surprising was
        gae = delta + gamma * lam * masks[step] * gae
        advantages.insert(0, gae)
    returns = [a + v for a, v in zip(advantages, values)]
    return advantages, returns

In [20]:
batch_size = 2048
minibatch_size = 512
update_epochs = 4
clip_eps = 0.2

In [40]:
import gymnasium
import torch

num_envs = 16
num_steps = 128

env = gymnasium.make_vec(
    "CartPole-v1",
    num_envs=num_envs,
    vectorization_mode="sync",
    vector_kwargs={"autoreset_mode": gymnasium.vector.AutoresetMode.SAME_STEP},
)
# store the observations, actions, logprobs, rewards, dones, and values for each step in the rollout
obs_dim = env.single_observation_space.shape[0]
obs_buf      = torch.zeros((num_steps, num_envs, obs_dim))
actions_buf  = torch.zeros((num_steps, num_envs))
logprobs_buf = torch.zeros((num_steps, num_envs))
rewards_buf  = torch.zeros((num_steps, num_envs))
dones_buf    = torch.zeros((num_steps, num_envs))
values_buf   = torch.zeros((num_steps, num_envs))

# initialize the actor/critic networks
actor = Actor(state_size, action_size)
critic = Critic(state_size)

next_obs, _ = env.reset()
next_obs = torch.tensor(next_obs, dtype=torch.float32)

def collect_rollout():
    global next_obs

    for step in range(num_steps):
        obs_buf[step] = next_obs

        with torch.no_grad():
            distribution = actor(next_obs)
            action = distribution.sample()
            logprob = distribution.log_prob(action)
            value = torch.squeeze(critic(next_obs), dim=-1)

        actions_buf[step] = action
        logprobs_buf[step] = logprob
        values_buf[step] = value

        # act in the environment and then get the next observation based on our predicted action
        next_obs, reward, done, truncated, info = env.step(action.numpy())

        rewards_buf[step] = torch.tensor(reward, dtype=torch.float32)
        dones_buf[step] = torch.tensor(done | truncated, dtype=torch.float32)

        next_obs = torch.tensor(next_obs, dtype=torch.float32)

    with torch.no_grad():
        next_value = torch.squeeze(critic(next_obs), dim=-1)

    return next_value

In [42]:
gae_buf = torch.zeros((num_steps, num_envs))
next_value = collect_rollout()



tensor([0.0570, 0.0834, 0.0523, 0.0440, 0.0513, 0.0651, 0.0784, 0.0824, 0.0375,
        0.0210, 0.0737, 0.0785, 0.0692, 0.0539, 0.0927, 0.0534])